# 03 — How a network learns: backpropagation

Last notebook we *set* a network's weights by hand to solve XOR. That left the real question open:
how does a network **find** good weights on its own? The answer is gradient descent — the very loop
you built in chapter 03 — paired with one idea that carries the gradient through a hidden layer: the
**chain rule**, applied layer by layer. That pairing is **backpropagation**, and we build it here by
hand, then check it against the library.

**Prerequisites**
- 11.1 — the artificial neuron == logistic regression; 11.2 — the hidden layer and why a
  non-linearity is essential.
- Chapter 03 — the log-loss, the single-neuron gradient `(P − y)·x`, and the update `θ ← θ − η·∇L`.

**What you'll be able to do**
- Run a forward pass that **caches** the hidden activation.
- Derive and code the backward pass — the chain rule, three links.
- Gradient-check it against finite differences (to machine precision).
- Train a `2 → 4 → 1` network by gradient descent until it solves the circles.
- Explain why the weights must be initialized to **break symmetry**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
np.random.seed(SEED)

# The exact concentric circles from notebook 11.2, same stratified 75/25 split.
X, y = make_circles(n_samples=400, noise=0.10, factor=0.40, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
print(f"train {X_tr.shape}   test {X_te.shape}   train class balance {np.bincount(y_tr)}")

## Where we left off

In notebook 11.2 we hand-set the weights of a small network and watched it solve XOR — a problem one
neuron cannot. We chose those weights to prove it was possible. But a useful model has to *learn* its
weights from data. We already know the tool for one neuron (chapter 03): take the gradient of the
loss, step every weight a little downhill, repeat. The single missing piece is the gradient of a
**hidden** weight — a weight buried one layer deep, with the loss sitting two layers away. The chain
rule supplies exactly that.

Recall the chapter-03 ingredients we will reuse unchanged: the **log-loss** measures how wrong the
probabilities are; the single-neuron gradient is `(P − y)·x`; the update is `θ ← θ − η·∇L`.

## Two passes: forward caches, backward carries the error back

Our network is a chain of operations:

`x → (W₁, b₁) → σ → H → (W₂, b₂) → σ → P`.

To nudge an output-layer weight we need to know how the loss responds to it — and `P` sits right
there, so that part is chapter 03 over again. To nudge a **hidden**-layer weight in `W₁`, the loss is
two steps downstream. The chain rule walks those steps **backward**:

- **Forward pass:** compute the activations left to right, and **cache** the hidden activation `H` —
  we will need it on the way back.
- **Backward pass:** start from the output error, pull it back through `W₂`, gate it by how
  responsive the hidden layer was, and arrive at `W₁`.

Let's write the forward pass first.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def forward(params, X):
    """Forward pass of a 2 -> H -> 1 sigmoid network; returns the cached activations."""
    W1, b1, W2, b2 = params
    Z1 = X @ W1 + b1       # (n, H)  hidden pre-activation
    H = sigmoid(Z1)        # (n, H)  hidden activation  <- cached for the backward pass
    Z2 = H @ W2 + b2       # (n, 1)  output pre-activation
    P = sigmoid(Z2)        # (n, 1)  probability of class 1
    return Z1, H, Z2, P


def log_loss(P, y):
    """Mean binary cross-entropy (the chapter-03 loss)."""
    yc = y.reshape(-1, 1).astype(float)
    Pc = np.clip(P, 1e-12, 1 - 1e-12)
    return float(-np.mean(yc * np.log(Pc) + (1 - yc) * np.log(1 - Pc)))


def make_net(seed, hidden=4, scale=0.7):
    """Random initial weights for a 2 -> hidden -> 1 network."""
    r = np.random.default_rng(seed)
    return [r.standard_normal((2, hidden)) * scale, np.zeros(hidden),
            r.standard_normal((hidden, 1)) * scale, np.zeros(1)]


params = make_net(SEED)
Z1, H, Z2, P = forward(params, X_tr)
print("W1 shape", params[0].shape, "  W2 shape", params[2].shape)
print("cached hidden H", H.shape, "   output P", P.shape)
print("first 3 probabilities:", P[:3, 0].round(3).tolist())
print(f"log-loss at initialization = {log_loss(P, y_tr):.4f}")

## The chain rule, written once (a picture, not a grind)

The loss sits at the end of the chain `x → Z₁ → H → Z₂ → P → loss`. The chain rule gives us each
weight's gradient by walking that chain **backwards**, one link at a time:

1. **Output error.** Exactly as in chapter 03, the sigmoid-and-log-loss gradient at the output is
   `d_out = P − y`. (We average the loss over the `n` examples, so a factor `1/n` rides along; we fold
   it into `d_out` to keep the rest clean.)
2. **Pull it back through W₂.** The hidden layer's share of the error is `d_out · W₂ᵀ` — the output
   error routed back along the same weights it flowed forward through.
3. **Gate by the hidden slope.** A hidden unit passes error in proportion to how responsive it was,
   so we multiply by the sigmoid derivative `σ′(Z₁) = H·(1 − H)`. Together:
   `dH = (d_out · W₂ᵀ) ⊙ H(1 − H)`.

Each weight gradient is then **error × input** at that layer: `dW₂ = Hᵀ d_out` and `dW₁ = Xᵀ dH`
(the biases are the errors summed over the examples). That is the whole of backpropagation for this
network.

In [ ]:
def backward(params, X, y):
    """Analytic gradient of the mean log-loss via the chain rule (1/n folded into d_out)."""
    W1, b1, W2, b2 = params
    yc = y.reshape(-1, 1).astype(float)
    _, H, _, P = forward(params, X)        # forward pass, re-using the cached H
    n = X.shape[0]

    d_out = (P - yc) / n                    # (n, 1)  output error  (chapter-03 gradient, averaged)
    dW2 = H.T @ d_out                       # (H, 1)  error x input at the output layer
    db2 = d_out.sum(axis=0)                 # (1,)
    dH = (d_out @ W2.T) * (H * (1 - H))      # (n, H)  pull back through W2, gate by H(1-H)
    dW1 = X.T @ dH                          # (2, H)  error x input at the hidden layer
    db1 = dH.sum(axis=0)                    # (H,)
    return [dW1, db1, dW2, db2]


grads = backward(params, X_tr, y_tr)
for name, g in zip(["dW1", "db1", "dW2", "db2"], grads, strict=True):
    print(f"{name}: gradient shape {g.shape}")

## Is our gradient correct? Gradient checking

A hand-derived gradient is easy to get subtly wrong. There is a slow-but-foolproof check: nudge one
weight up by a tiny `ε` and down by `ε`, measure how much the loss actually changes, and divide. That
central finite difference *is* the gradient with respect to that weight — no calculus required. If it
agrees with our analytic gradient to near machine precision, the chain-rule code is right. This is the
same gradient-check every deep-learning library is validated against.

In [ ]:
def numerical_gradient(params, X, y, eps=1e-6):
    """Brute-force gradient by central finite differences, one parameter at a time."""
    nums = []
    for p in params:
        gp = np.zeros_like(p)
        it = np.nditer(p, flags=["multi_index"])
        while not it.finished:
            idx = it.multi_index
            orig = p[idx]
            p[idx] = orig + eps
            loss_plus = log_loss(forward(params, X)[3], y)
            p[idx] = orig - eps
            loss_minus = log_loss(forward(params, X)[3], y)
            p[idx] = orig
            gp[idx] = (loss_plus - loss_minus) / (2 * eps)
            it.iternext()
        nums.append(gp)
    return nums


analytic = backward(params, X_tr, y_tr)
numeric = numerical_gradient(params, X_tr, y_tr)
flat_a = np.concatenate([g.ravel() for g in analytic])
flat_n = np.concatenate([g.ravel() for g in numeric])
rel_err = np.linalg.norm(flat_a - flat_n) / (np.linalg.norm(flat_a) + np.linalg.norm(flat_n))
print(f"parameters checked : {flat_a.size}")
print(f"relative error     : {rel_err:.2e}")

**Read the result.** The hand-derived gradient and the brute-force numerical one agree to about ten
significant digits — a relative error on the order of `1e-10`, near the round-off floor of 64-bit
arithmetic. The chain-rule code is correct. With a gradient we can trust, training is now within
reach.

In [ ]:
C = colors.COLORS
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis("off")

inputs = [(0.0, 2.2, "x₁"), (0.0, 0.8, "x₂")]
hidden = [(2.2, hy) for hy in (3.0, 2.2, 1.4, 0.6)]
out = (4.4, 1.8)

for ix, iy, _ in inputs:                         # forward connections (input -> hidden)
    for hx, hy in hidden:
        ax.plot([ix, hx], [iy, hy], color=C["muted"], lw=0.7, zorder=1)
for hx, hy in hidden:                            # forward connections (hidden -> output)
    ax.plot([hx, out[0]], [hy, out[1]], color=C["muted"], lw=0.7, zorder=1)

for ix, iy, label in inputs:
    ax.add_patch(plt.Circle((ix, iy), 0.18, color=C["class_a"], ec=C["text"], lw=1.2, zorder=3))
    ax.text(ix, iy, label, ha="center", va="center", color=C["text"], zorder=4)
for hx, hy in hidden:
    ax.add_patch(plt.Circle((hx, hy), 0.18, color=C["class_c"], ec=C["text"], lw=1.2, zorder=3))
ax.add_patch(plt.Circle(out, 0.20, color=C["class_b"], ec=C["text"], lw=1.2, zorder=3))
ax.text(out[0], out[1], "P", ha="center", va="center", color=C["text"], zorder=4)

ax.text(1.1, 3.6, "H = σ(X·W₁ + b₁)", color=C["model"], ha="center", fontsize=10)
ax.text(3.3, 3.6, "P = σ(H·W₂ + b₂)", color=C["model"], ha="center", fontsize=10)

ax.annotate("", xy=(4.7, 4.2), xytext=(-0.3, 4.2),
            arrowprops=dict(arrowstyle="->", color=C["model"], lw=2))
ax.text(2.2, 4.38, "forward — compute and cache H", color=C["model"], ha="center", fontsize=11)
ax.annotate("", xy=(-0.3, -0.7), xytext=(4.7, -0.7),
            arrowprops=dict(arrowstyle="->", color=C["highlight"], lw=2))
ax.text(2.2, -1.05, "backward — the chain rule sends the error back", color=C["highlight"],
        ha="center", fontsize=11)
ax.text(4.4, 0.95, "d_out = P − y", color=C["highlight"], ha="center", fontsize=9)
ax.text(2.0, -0.15, "dH = (d_out·W₂ᵀ) ⊙ H(1−H)", color=C["highlight"], ha="center", fontsize=9)

ax.set_xlim(-0.9, 5.3)
ax.set_ylim(-1.3, 4.6)
ax.set_title("Backpropagation: one forward pass, one backward pass")
plt.show()

**Read the figure.** The forward pass (top, sky) fills in the activations left to right and stores the
hidden layer `H`. The backward pass (bottom, rose) runs the other way: it starts from the output error
`d_out = P − y`, pulls it back through `W₂`, and gates it by the hidden slope `H(1 − H)` to get `dH`.
Two passes — one to compute, one to assign blame — and the same two passes work for a network of any
depth.

## Now descend

With a correct gradient, training is the chapter-03 loop, unchanged: compute the gradient, step every
weight downhill by `θ ← θ − η·∇L`, and repeat. The only thing that is new is *where the gradient comes
from* — backpropagation instead of a one-line formula. Let's run it on the circles and watch the loss
fall.

In [ ]:
def accuracy(params, X, y):
    return ((forward(params, X)[3].ravel() > 0.5).astype(int) == y).mean()


def train_gd(start, lr, iters):
    """Full-batch gradient descent; returns the trained params and the loss per iteration."""
    p = [a.copy() for a in start]
    losses = []
    for _ in range(iters):
        g = backward(p, X_tr, y_tr)
        for k in range(len(p)):
            p[k] = p[k] - lr * g[k]
        losses.append(log_loss(forward(p, X_tr)[3], y_tr))
    return p, losses


LR, ITERS = 3.0, 8000
trained, losses = train_gd(make_net(SEED), LR, ITERS)
print(f"by-hand 2-4-1 (GD): train acc {accuracy(trained, X_tr, y_tr):.2f}   "
      f"test acc {accuracy(trained, X_te, y_te):.2f}   final loss {losses[-1]:.4f}")

mlp = MLPClassifier(hidden_layer_sizes=(4,), activation="logistic", solver="lbfgs",
                    max_iter=5000, random_state=SEED).fit(X_tr, y_tr)
print(f"MLPClassifier (lbfgs): train acc {mlp.score(X_tr, y_tr):.2f}   "
      f"test acc {mlp.score(X_te, y_te):.2f}")


class HandNet:
    """Wrap the by-hand params so viz.plot_decision_boundary can call .predict."""

    def __init__(self, params):
        self.params = params

    def predict(self, X):
        return (forward(self.params, np.asarray(X, dtype=float))[3].ravel() > 0.5).astype(int)


C = colors.COLORS
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].plot(range(1, ITERS + 1), losses, color=C["model"], lw=2)
axes[0].set_xlabel("gradient-descent iteration")
axes[0].set_ylabel("training log-loss")
axes[0].set_title(f"By-hand backprop + GD (train acc {accuracy(trained, X_tr, y_tr):.2f})")

viz.plot_decision_boundary(HandNet(trained), X_te, y_te, ax=axes[1])
pad = 0.5
x0, x1 = X_te[:, 0].min() - pad, X_te[:, 0].max() + pad
y0, y1 = X_te[:, 1].min() - pad, X_te[:, 1].max() + pad
gx, gy = np.meshgrid(np.linspace(x0, x1, 300), np.linspace(y0, y1, 300))
zz = mlp.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
axes[1].contour(gx, gy, zz, levels=[0.5], colors=[C["highlight"]], linestyles="--", linewidths=2)
axes[1].plot([], [], color=C["highlight"], ls="--", lw=2, label="MLPClassifier (lbfgs)")
axes[1].legend(loc="upper right")
axes[1].set_title("By-hand regions vs library boundary (dashed)")
plt.show()

**Read the figure.** Left: the training loss falls steadily toward zero as gradient descent runs,
and the by-hand network reaches **train and test accuracy 1.0** — it has carved the closed ring
that a single neuron (notebook 11.2) could not. Right: `MLPClassifier`, the library's own
implementation, finds the **same closed ring** (its dashed boundary traces ours); it reaches
comparable accuracy (here 0.98 / 0.95). The exact number depends on where the optimizer starts: across
random initializations the library lands anywhere from 0.85 to 1.0, because a multi-layer loss is
**non-convex** — it has many good minima rather than one. That is a first glimpse of something
notebook 11.4 takes up directly; here the lesson is that the library automates the very forward pass,
backward pass, and descent we wrote by hand.

*(We trained with full-batch gradient descent and read the library's `lbfgs` result because on this
sigmoid network the mini-batch solvers `adam`/`sgd` stall near chance — a saturation effect that ReLU
resolves in notebook 11.4.)*

## Why the starting weights must break symmetry

One detail of training is easy to overlook and quietly fatal: the *initial* weights. Suppose two
hidden units start with identical weights. They then compute the same thing on every input, so the
chain rule hands them the **same** gradient, so they take the **same** step — and they stay identical
forever. The network has four hidden units but an effective width of one. The cure is to start each
unit from a different random point, so their gradients differ and they specialize. Let's see all of
this happen.

In [ ]:
def init_zeros(hidden=4):
    return [np.zeros((2, hidden)), np.zeros(hidden), np.zeros((hidden, 1)), np.zeros(1)]


def init_symmetric(seed, hidden=4, scale=0.7):
    """Every hidden unit identical: one shared incoming column, one shared outgoing weight."""
    r = np.random.default_rng(seed)
    shared = r.standard_normal((2, 1)) * scale
    W1 = np.repeat(shared, hidden, axis=1)
    W2 = np.full((hidden, 1), float(r.standard_normal()) * scale)
    return [W1, np.zeros(hidden), W2, np.zeros(1)]


def col_spread(W1):
    """Mean spread across the hidden columns; 0 means all units are identical."""
    return float(W1.std(axis=1).mean())


SYM_LR, SYM_ITERS = 1.0, 15000
starts = {"full zeros": init_zeros(),
          "identical units": init_symmetric(SEED),
          "random": make_net(SEED)}
results = {}
for name, start in starts.items():
    g0 = backward(start, X_tr, y_tr)
    grad_norm = float(np.sqrt(sum(np.sum(gi ** 2) for gi in g0)))
    p, curve = train_gd(start, SYM_LR, SYM_ITERS)
    results[name] = (p, curve)
    print(f"{name:16s}: |grad@init|={grad_norm:.4f}   train acc {accuracy(p, X_tr, y_tr):.2f}   "
          f"col-spread {col_spread(start[0]):.3f} -> {col_spread(p[0]):.3f}")

C = colors.COLORS
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

sym_W1 = results["identical units"][0][0]
rnd_W1 = results["random"][0][0]
axes[0].scatter(sym_W1[0], sym_W1[1], s=170, marker="X", color=C["error"],
                edgecolor=C["text"], zorder=3, label="identical units (4 on one point)")
axes[0].scatter(rnd_W1[0], rnd_W1[1], s=120, marker="o", color=C["model"],
                edgecolor=C["text"], zorder=3, label="random init (4 distinct)")
axes[0].axhline(0, color=C["grid"], lw=1)
axes[0].axvline(0, color=C["grid"], lw=1)
axes[0].set_xlabel("incoming weight on x₁")
axes[0].set_ylabel("incoming weight on x₂")
axes[0].set_title("Hidden units after training")
axes[0].legend(loc="best")

curve_color = {"full zeros": C["error"], "identical units": C["muted"], "random": C["model"]}
for name, (_, curve) in results.items():
    axes[1].plot(range(1, SYM_ITERS + 1), curve, color=curve_color[name], lw=2, label=name)
axes[1].set_xlabel("gradient-descent iteration")
axes[1].set_ylabel("training log-loss")
axes[1].set_title("Loss by initialization")
axes[1].legend(loc="best")
plt.show()

**Read the figure.** Three initializations, three fates. **Full zeros**: here every gradient is
exactly zero — on balanced data the output error averages to zero, and `W₂ = 0` zeroes the rest — so
the loss never leaves `ln 2 ≈ 0.6931` and accuracy sits at 0.50; the network is frozen.
**Identical units**: the four hidden units share one starting point, receive identical gradients, and
never separate — their weight vectors stay stacked on a single point (left panel) and accuracy is
stuck at 0.68, the ceiling of an effective width of one. **Random init**: the units start apart,
specialize into four distinct vectors, and the loss drives to zero (accuracy 1.0). Random
initialization is not a detail — it is what lets a hidden layer become more than a single neuron.

## The same idea, at any depth

Nothing above was special to two layers. For a network of any depth, the two passes are identical in
spirit:

- **Forward:** push the input through each layer, **caching** every activation along the way.
- **Backward:** start with the output error and send it back through each layer in turn — route it
  through that layer's weights (`Wᵀ`), then gate it by the activation's local slope `φ′(z)`.

Written compactly, the error at layer `l` is `δₗ = (Wₗ₊₁ᵀ · δₗ₊₁) ⊙ φ′(zₗ)` — the same quantity we
called `dH` above, now written `δ` for an arbitrary layer `l` — and each weight gradient is that
layer's error times its input. In words: *every layer's error is the next layer's error,
pulled back through the weights and scaled by how active this layer was.* That recursion is
backpropagation (Rumelhart, Hinton & Williams, 1986) — the engine under every network in this course.

## Your turn

1. **(warm-up)** Re-train the by-hand network with `LR = 0.1`, then with `LR = 30`. Predict what each
   loss curve will look like *before* running, then check: which one crawls, and which one becomes
   unstable?
2. **(core)** In `init_symmetric`, make only the **output** weights `W2` identical while `W1` stays
   random. Does symmetry still break? You measured the mechanism above — which weights must differ for
   the units to specialize?
3. **(reach)** Add a second hidden layer (a `2 → 4 → 4 → 1` network). Extend the forward pass to cache
   both hidden activations and the backward pass to carry the error back one more link, then
   gradient-check your new `backward` the same way we did here.

## What you built

- You wrote a forward pass that **caches** the hidden activation, then derived and coded the backward
  pass — the **chain rule** in three links (`d_out → dH → dW₁, dW₂`).
- You **gradient-checked** it against finite differences and matched to about ten digits — proof the
  math is right.
- You trained a `2 → 4 → 1` network by gradient descent to **train and test accuracy 1.0** on the
  circles, the closed boundary a single neuron could not draw.
- You saw why initialization must **break symmetry**: zeros freeze, identical units stay identical,
  random init lets them specialize.

You now have backpropagation by hand. Next: the real `MLPClassifier` — its parameters, the **ReLU**
activation, and the **K-class softmax** output head.

## References

- Rumelhart, D. E., Hinton, G. E. & Williams, R. J. (1986). *Learning representations by
  back-propagating errors.* Nature 323, 533–536. https://doi.org/10.1038/323533a0
- LeCun, Y., Bottou, L., Orr, G. B. & Müller, K.-R. (1998). *Efficient BackProp.* In Neural Networks:
  Tricks of the Trade. https://doi.org/10.1007/3-540-49430-8_2 (gradient checking; practical backprop).
- Glorot, X. & Bengio, Y. (2010). *Understanding the difficulty of training deep feedforward neural
  networks.* PMLR 9, 249–256 (initialization and symmetry breaking).
- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, §6.5 (back-propagation).
  MIT Press.

*Previous:* 11.2 — the hidden layer.  *Next:* **11.4 — the estimator `MLPClassifier` and its
parameters.**